# Class 11 — classes in practice: solutions

This notebook contains solutions to the exercises of Class 11 (class methods, properties, when to use a class, inheritance and composition). It is self-contained: it starts from class `Tray` of Class 9 (the script `class11/nursery.py`), and changes it step by step, as in the class. Each time a class changes, the whole class is written again, so that each cell can be read on its own.

Run the cells in order: later cells use the classes defined in earlier cells.

The solution of the *Cookie jar* assignment (5.5) is not included.

---
# Starting point: class `Tray` of Class 9

Class `Tray` of Class 9 (3.1 and 3.2), as in the script `class11/nursery.py`.

In [ ]:
class Tray:
    MAX_PLANTS = 50   # class attribute: shared by all trays

    def __init__(self, species, n_plants=0):
        if n_plants < 0 or n_plants > Tray.MAX_PLANTS:
            raise ValueError(f"n_plants must be between 0 and {Tray.MAX_PLANTS}")
        self.species = species
        self.n_plants = n_plants

    def add_plants(self, n):
        if self.n_plants + n > Tray.MAX_PLANTS:
            raise ValueError(f"there is only room for {Tray.MAX_PLANTS - self.n_plants} more plants")
        self.n_plants += n

    def remove_plants(self, n):
        if n > self.n_plants:
            raise ValueError(f"the tray only has {self.n_plants} plants")
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants"

---
# Part 1 — Revision, and class methods

## Exercise 1.2

`n_trays` is a class variable: there is a single counter, shared by all trays, stored in the class. `__init__` increases it each time a tray is created.

The line `Tray.n_trays += 1` is at the **end** of `__init__`, after the validation: if `n_plants` is not valid, `raise` stops `__init__` before the counter is increased, so trays that could not be created are not counted.

In [ ]:
class Tray:
    MAX_PLANTS = 50   # class variable: the same for all trays
    n_trays = 0       # class variable: number of trays created

    def __init__(self, species, n_plants=0):
        if n_plants < 0 or n_plants > Tray.MAX_PLANTS:
            raise ValueError(f"n_plants must be between 0 and {Tray.MAX_PLANTS}")
        self.species = species
        self.n_plants = n_plants
        Tray.n_trays += 1   # at the end: only trays that were created are counted

    def add_plants(self, n):
        if self.n_plants + n > Tray.MAX_PLANTS:
            raise ValueError(f"there is only room for {Tray.MAX_PLANTS - self.n_plants} more plants")
        self.n_plants += n

    def remove_plants(self, n):
        if n > self.n_plants:
            raise ValueError(f"the tray only has {self.n_plants} plants")
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants"

In [ ]:
tray1 = Tray("Pansy", 20)
tray2 = Tray("Petunia", 30)
tray3 = Tray("Begonia")
print(Tray.n_trays)          # 3

try:
    Tray("Pansy", 60)        # not valid: not counted
except ValueError as e:
    print("ValueError:", e)
print(Tray.n_trays)          # still 3

Note that the counter must be increased with `Tray.n_trays += 1`, and not with `self.n_trays += 1`: assigning to `self.n_trays` would create a new attribute of **each** tray, equal to 1, and the counter of the class would stay at 0.

## 1.3 and Exercise 1.4: class methods

Class `Tray` with the counter of 1.2, the class method `report` of 1.3, and the class method `from_csv_row` of 1.4. Both class methods receive the class as `cls`, and are called on the class: `Tray.report()`, `Tray.from_csv_row(...)`.

In [ ]:
class Tray:
    MAX_PLANTS = 50
    n_trays = 0

    def __init__(self, species, n_plants=0):
        if n_plants < 0 or n_plants > Tray.MAX_PLANTS:
            raise ValueError(f"n_plants must be between 0 and {Tray.MAX_PLANTS}")
        self.species = species
        self.n_plants = n_plants
        Tray.n_trays += 1

    @classmethod
    def report(cls):                                  # 1.3
        print(f"{cls.n_trays} trays created, at most {cls.MAX_PLANTS} plants each")

    @classmethod
    def from_csv_row(cls, row):                       # 1.4
        species, n_plants = row.split(",")
        return cls(species.strip(), int(n_plants))   # cls(...) is Tray(...)

    def add_plants(self, n):
        if self.n_plants + n > Tray.MAX_PLANTS:
            raise ValueError(f"there is only room for {Tray.MAX_PLANTS - self.n_plants} more plants")
        self.n_plants += n

    def remove_plants(self, n):
        if n > self.n_plants:
            raise ValueError(f"the tray only has {self.n_plants} plants")
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants"

In [ ]:
Tray.report()                      # no tray yet: called on the class

tray1 = Tray.from_csv_row("Petunia,30")
tray2 = Tray.from_csv_row("Pansy, 25")
print(tray1)
print(tray2)
Tray.report()

`from_csv_row` could not be an instance method: an instance method is called on an existing tray (`self`), but the purpose of `from_csv_row` is to **create** a tray, so there is no tray yet to call it on. We would have to create a dummy tray first, only to call the method, which makes no sense.

---
# Part 2 — Properties

## Exercise 2.2

The class of section 2.2: `n_plants` is a property, with a getter and a setter.

In [ ]:
class Tray:
    MAX_PLANTS = 50

    def __init__(self, species, n_plants=0):
        self.species = species
        self.n_plants = n_plants        # calls the setter: the value is checked

    @property
    def n_plants(self):                 # getter: runs on tray.n_plants
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):          # setter: runs on tray.n_plants = value
        if value < 0 or value > Tray.MAX_PLANTS:
            raise ValueError(f"n_plants must be between 0 and {Tray.MAX_PLANTS}")
        self._n_plants = value

    def add_plants(self, n):
        self.n_plants += n

    def remove_plants(self, n):
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants"

In [ ]:
tray = Tray("Pansy", 20)

try:
    tray.n_plants = -5
except ValueError as e:
    print("ValueError:", e)

try:
    tray.add_plants(100)
except ValueError as e:
    print("ValueError:", e)

try:
    Tray("Pansy", 60)
except ValueError as e:
    print("ValueError:", e)

print(tray)          # the tray did not change
print(vars(tray))    # the number of plants is stored in _n_plants

The three changes are checked by the setter, and `tray` keeps its 20 plants. `vars(tray)` shows that the tray has an attribute `_n_plants`, and no attribute `n_plants`: `n_plants` is a property of the class, which reads and writes `_n_plants`.

If the setter does `self.n_plants = value` instead of `self._n_plants = value`, that assignment calls the setter again, which calls the setter again, and so on: Python stops with a `RecursionError` when the tray is created.

In [ ]:
class BrokenTray:
    def __init__(self, n_plants):
        self.n_plants = n_plants

    @property
    def n_plants(self):
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):
        self.n_plants = value           # calls this setter again, forever

try:
    BrokenTray(20)
except RecursionError as e:
    print("RecursionError:", e)

## 2.3 Read-only and computed properties: trays of different sizes

The class of section 2.3, with `from_csv_row` of section 1.4:

In [ ]:
class Tray:
    CAPACITIES = (24, 50, 104)          # class variable: standard tray sizes

    def __init__(self, species, n_plants=0, capacity=50):
        if capacity not in Tray.CAPACITIES:
            raise ValueError(f"capacity must be one of {Tray.CAPACITIES}")
        self._capacity = capacity       # must come before n_plants (see the setter)
        self.species = species
        self.n_plants = n_plants        # calls the setter below

    @classmethod
    def from_csv_row(cls, row):
        species, n_plants = row.split(",")
        return cls(species.strip(), int(n_plants))

    @property
    def capacity(self):                 # read-only: there is no setter
        return self._capacity

    @property
    def n_plants(self):                 # getter: runs on tray.n_plants
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):          # setter: runs on tray.n_plants = value
        if value < 0:
            raise ValueError("n_plants cannot be negative")
        if value > self.capacity:
            raise ValueError(f"the tray only has {self.capacity} cells")
        self._n_plants = value

    @property
    def free_cells(self):               # computed from other attributes
        return self.capacity - self.n_plants

    def add_plants(self, n):
        self.n_plants += n              # also goes through the setter

    def remove_plants(self, n):
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants ({self.free_cells} free cells)"

## Exercise 2.3

`type(e).__name__` gives the name of the class of the exception `e`.

In [ ]:
tray = Tray("Pansy", 20)

try:
    Tray("Pansy", 30, capacity=24)
except Exception as e:
    print(type(e).__name__, "-", e)

try:
    tray.n_plants = -5
except Exception as e:
    print(type(e).__name__, "-", e)

try:
    tray.add_plants(100)
except Exception as e:
    print(type(e).__name__, "-", e)

try:
    tray.capacity = 100
except Exception as e:
    print(type(e).__name__, "-", e)

print(tray)   # the tray did not change

The first three raise `ValueError`, from the setter of `n_plants`, which runs in `__init__`, in `tray.n_plants = -5`, and in `add_plants`. The last one raises `AttributeError`, because the property `capacity` has no setter.

## Exercises 2.4 and 2.5

- 2.4: `from_csv_row` reads three values, and passes the capacity to `cls(...)`.
- 2.5: `species` becomes a property. Its setter runs in `__init__` (`self.species = species`), so invalid species are rejected when the tray is created. `isinstance(value, str)` checks that the value is a string.

In [ ]:
class Tray:
    CAPACITIES = (24, 50, 104)

    def __init__(self, species, n_plants=0, capacity=50):
        if capacity not in Tray.CAPACITIES:
            raise ValueError(f"capacity must be one of {Tray.CAPACITIES}")
        self._capacity = capacity
        self.species = species
        self.n_plants = n_plants

    @classmethod
    def from_csv_row(cls, row):                                     # 2.4
        species, n_plants, capacity = row.split(",")
        return cls(species.strip(), int(n_plants), int(capacity))

    @property
    def species(self):                                              # 2.5
        return self._species

    @species.setter
    def species(self, value):                                       # 2.5
        if not isinstance(value, str) or value.strip() == "":
            raise ValueError("species must be a non-empty string")
        self._species = value.strip()

    @property
    def capacity(self):
        return self._capacity

    @property
    def n_plants(self):
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):
        if value < 0:
            raise ValueError("n_plants cannot be negative")
        if value > self.capacity:
            raise ValueError(f"the tray only has {self.capacity} cells")
        self._n_plants = value

    @property
    def free_cells(self):
        return self.capacity - self.n_plants

    def add_plants(self, n):
        self.n_plants += n

    def remove_plants(self, n):
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants ({self.free_cells} free cells)"

With the new `from_csv_row`, a list comprehension creates a list of trays from a list of rows:

In [ ]:
rows = ["Pansy,25,50", "Petunia,24,24", "Begonia,0,104"]
trays = [Tray.from_csv_row(row) for row in rows]
for tray in trays:
    print(tray)

In [ ]:
for species in ["", "   ", 42]:
    try:
        Tray(species, 10)
    except ValueError as e:
        print(repr(species), "-", e)

---
# Part 3 — When to use a class

## Exercise 3.1

The two classes become two functions, in a module `units.py`. In a script, you would create the file `units.py` in the editor. Here, the notebook writes the file, and then imports it.

In [ ]:
code = """def kg_to_tons(kg):
    return kg / 1000


def ha_to_m2(ha):
    return ha * 10000


def germination_rate(n_seeds, n_germinated):
    return n_germinated / n_seeds
"""
with open("units.py", "w") as file:
    file.write(code)

In [ ]:
import units

print(units.kg_to_tons(2500))
print(units.ha_to_m2(1.5))
print(units.germination_rate(100, 87))

Compare with the class versions: `Converter().kg_to_tons(2500)` and `GerminationRate(100, 87).compute()`. The functions are shorter, and they do the same thing.

## Exercise 3.2

| case | class or function? | why |
|---|---|---|
| (a) convert °F to °C | function | one computation, no data to keep: `(f - 32) * 5 / 9` |
| (b) greenhouse with sensors and alarms | class | several sensors, each with its own readings, which change over time; several operations (add a reading, compute the mean, check the alarm). A class `Sensor`, and a class `Greenhouse` that *has* sensors (composition) |
| (c) area of a field | function | one computation from two numbers |
| (d) seed bank | class | several seed lots, each with its own species, quantity and expiry date; lots are added, withdrawn and expire, and quantities must stay valid. A class `SeedLot`, and a class `SeedBank` that *has* lots, like `Inventory` and `Tray` in Class 9 |

---
# Part 4 — Inheritance, used sparingly, and composition

## Exercise 4.1

`CapacityError` is raised in the setter of `n_plants` when there are too many plants. A negative number of plants still raises `ValueError`.

In [ ]:
class CapacityError(ValueError):
    """Raised when a tray does not have room for the plants."""


class Tray:
    CAPACITIES = (24, 50, 104)

    def __init__(self, species, n_plants=0, capacity=50):
        if capacity not in Tray.CAPACITIES:
            raise ValueError(f"capacity must be one of {Tray.CAPACITIES}")
        self._capacity = capacity
        self.species = species
        self.n_plants = n_plants

    @classmethod
    def from_csv_row(cls, row):
        species, n_plants, capacity = row.split(",")
        return cls(species.strip(), int(n_plants), int(capacity))

    @property
    def species(self):
        return self._species

    @species.setter
    def species(self, value):
        if not isinstance(value, str) or value.strip() == "":
            raise ValueError("species must be a non-empty string")
        self._species = value.strip()

    @property
    def capacity(self):
        return self._capacity

    @property
    def n_plants(self):
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):
        if value < 0:
            raise ValueError("n_plants cannot be negative")
        if value > self.capacity:
            raise CapacityError(f"the tray only has {self.capacity} cells")   # 4.1
        self._n_plants = value

    @property
    def free_cells(self):
        return self.capacity - self.n_plants

    def add_plants(self, n):
        self.n_plants += n

    def remove_plants(self, n):
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants ({self.free_cells} free cells)"

In [ ]:
tray = Tray("Pansy", 20)

try:
    tray.add_plants(100)
except ValueError as e:
    print(type(e).__name__, isinstance(e, ValueError), isinstance(e, CapacityError))

try:
    tray.n_plants = -5
except ValueError as e:
    print(type(e).__name__, isinstance(e, ValueError), isinstance(e, CapacityError))

- `tray.add_plants(100)` raises a `CapacityError`, which *is a* `ValueError`: both `isinstance` return `True`, and `except ValueError` catches it.
- `tray.n_plants = -5` raises a `ValueError`, which is not a `CapacityError`.

`except CapacityError` catches only the first one:

In [ ]:
try:
    tray.n_plants = -5
except CapacityError:
    print("not enough room")
except ValueError:
    print("some other problem with the number of plants")

## 4.2 A child class with `super()`

In [ ]:
class SeedTray(Tray):
    def __init__(self, species, n_seeds=0, capacity=50, germination_rate=0.8):
        super().__init__(species, n_seeds, capacity)   # runs __init__ of Tray
        self.germination_rate = germination_rate

    def expected_plants(self):
        return round(self.n_plants * self.germination_rate)

    def __str__(self):                                  # overrides __str__ of Tray
        return f"{super().__str__()}, {self.expected_plants()} expected to germinate"


seeds = SeedTray("Basil", 50, germination_rate=0.7)
print(seeds)
print(seeds.free_cells, seeds.capacity)                # inherited from Tray
print(isinstance(seeds, SeedTray), isinstance(seeds, Tray))

## Exercise 4.4

The loop prints each tray with the `__str__` of its own class: `Tray.__str__` for the two trays, and `SeedTray.__str__` for the seed tray (polymorphism). `sum(tray.n_plants ...)` works, since every tray has `n_plants`.

In [ ]:
trays = [Tray("Pansy", 25), SeedTray("Basil", 50, germination_rate=0.7), Tray("Petunia", 10, capacity=24)]
for tray in trays:
    print(tray)
print(sum(tray.n_plants for tray in trays))

In [ ]:
try:
    print(sum(tray.expected_plants() for tray in trays))
except AttributeError as e:
    print("AttributeError:", e)

It fails because `expected_plants` is defined only in `SeedTray`: the first tray of the list is a `Tray`, which has no such method. The solution is to add to class `Tray` a method with the same name, that returns `n_plants` (all the plants of an ordinary tray are expected to be sold):
```python
    def expected_plants(self):
        return self.n_plants
```
Then `SeedTray` overrides it, and the same call works for every tray. This method is included in the final version of `Tray`, in Part 5, where the total is computed.

---
# Part 5 — On your own

## Exercises 5.1 and 5.4

The final version of class `Tray`, with `expected_plants` (4.4):

- 5.1: `__add__` returns a **new** tray, with the plants of both trays and the capacity of the larger one. If the plants do not fit, `CapacityError` is raised by the setter of `n_plants`, when the new tray is created.
- 5.4: the attributes `material`, `max_uses` (`None` means no limit) and `n_uses`. In `add_plants`, `self.n_uses` is increased **after** the plants are added, so that a failed `add_plants` does not count as a use.

In [ ]:
class Tray:
    CAPACITIES = (24, 50, 104)

    def __init__(self, species, n_plants=0, capacity=50, material="plastic", max_uses=None):
        if capacity not in Tray.CAPACITIES:
            raise ValueError(f"capacity must be one of {Tray.CAPACITIES}")
        self._capacity = capacity
        self.species = species
        self.n_plants = n_plants
        self.material = material        # 5.4
        self.max_uses = max_uses        # 5.4: None means no limit
        self.n_uses = 0                 # 5.4

    @classmethod
    def from_csv_row(cls, row):
        species, n_plants, capacity = row.split(",")
        return cls(species.strip(), int(n_plants), int(capacity))

    @property
    def species(self):
        return self._species

    @species.setter
    def species(self, value):
        if not isinstance(value, str) or value.strip() == "":
            raise ValueError("species must be a non-empty string")
        self._species = value.strip()

    @property
    def capacity(self):
        return self._capacity

    @property
    def n_plants(self):
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):
        if value < 0:
            raise ValueError("n_plants cannot be negative")
        if value > self.capacity:
            raise CapacityError(f"the tray only has {self.capacity} cells")
        self._n_plants = value

    @property
    def free_cells(self):
        return self.capacity - self.n_plants

    def add_plants(self, n):
        if self.max_uses is not None and self.n_uses >= self.max_uses:     # 5.4
            raise ValueError(f"the tray was already used {self.max_uses} times")
        self.n_plants += n
        self.n_uses += 1                                                   # 5.4

    def remove_plants(self, n):
        self.n_plants -= n

    def expected_plants(self):                                             # 4.4
        return self.n_plants

    def __str__(self):
        return f"{self.n_plants} {self.species} plants ({self.free_cells} free cells)"

    def __add__(self, other):                                              # 5.1
        if self.species != other.species:
            raise ValueError("cannot merge trays of different species")
        capacity = max(self.capacity, other.capacity)
        return Tray(self.species, self.n_plants + other.n_plants, capacity)

In [ ]:
# 5.1
small = Tray("Pansy", 20, capacity=24)
large = Tray("Pansy", 30)
merged = small + large            # calls small.__add__(large)
print(merged)
print(small, "|", large)          # the original trays did not change

for a, b in [(Tray("Pansy"), Tray("Petunia")), (Tray("Pansy", 30), Tray("Pansy", 30))]:
    try:
        a + b
    except ValueError as e:
        print(type(e).__name__, "-", e)

In [ ]:
# 5.4
tray = Tray("Pansy", capacity=24, max_uses=2)
tray.add_plants(20)
tray.remove_plants(20)
tray.add_plants(24)
tray.remove_plants(24)
try:
    tray.add_plants(10)           # third use
except ValueError as e:
    print(e)
print(tray, "- used", tray.n_uses, "times")

unlimited = Tray("Pansy")          # max_uses=None
for _ in range(5):
    unlimited.add_plants(1)
print(unlimited, "- used", unlimited.n_uses, "times")

`SeedTray` must be defined again, so that it inherits from the new `Tray`. Then the total of 4.4 works for all trays:

In [ ]:
class SeedTray(Tray):
    def __init__(self, species, n_seeds=0, capacity=50, germination_rate=0.8):
        super().__init__(species, n_seeds, capacity)
        self.germination_rate = germination_rate

    def expected_plants(self):                          # overrides expected_plants of Tray
        return round(self.n_plants * self.germination_rate)

    def __str__(self):
        return f"{super().__str__()}, {self.expected_plants()} expected to germinate"


trays = [Tray("Pansy", 25), SeedTray("Basil", 50, germination_rate=0.7), Tray("Petunia", 10, capacity=24)]
print(sum(tray.expected_plants() for tray in trays))   # 25 + 35 + 10

## Exercises 5.2 and 5.3

These are **functions**, not methods: they work on a list of trays, and do not need to keep any data of their own (Part 3).

In `load_trays`, each non-empty line of the file (for instance `Pansy,25,50`) is passed to `Tray.from_csv_row`.

In [ ]:
def load_trays(path):                                   # 5.2
    trays = []
    with open(path) as file:
        for line in file:
            line = line.strip()
            if line != "":
                trays.append(Tray.from_csv_row(line))
    return trays


def free_cells(trays):                                  # 5.3
    return sum(tray.free_cells for tray in trays)


def full_trays(trays):                                  # 5.3
    return [tray for tray in trays if tray.free_cells == 0]

A small file to test `load_trays` (in the codespace, you can also create it in the editor):

In [ ]:
with open("trays.csv", "w") as file:
    file.write("Pansy,25,50\n")
    file.write("Pansy,50,50\n")
    file.write("Petunia,24,24\n")
    file.write("Begonia,0,104\n")

In [ ]:
trays = load_trays("trays.csv")
for tray in trays:
    print(tray)
print("free cells:", free_cells(trays))
print("full trays:")
for tray in full_trays(trays):
    print("  ", tray)